In [1]:
import os
import numpy as np
import tifffile

def find_best_crop_save_diff(large_path, small_path, output_dir, search_radius=3):
    """
    Finds the best crop (within ±search_radius pixels around the center) of the large TIFF image
    that best matches the small TIFF image. Saves the absolute difference image in TIFF format
    with a filename that includes the crop coordinates and the difference score.
    
    Parameters:
      large_path (str): Path to the large TIFF image.
      small_path (str): Path to the small TIFF image.
      output_dir (str): Directory where the difference image will be saved.
      search_radius (int): Maximum pixel offset (in each direction) from the center to search.
      
    Returns:
      best_coord (tuple): (top_left_x, top_left_y) of the best crop.
      crop_size (tuple): (width, height) of the crop.
      best_score (float): The sum of absolute differences score for the best match.
    """
    # Load images using tifffile
    large = tifffile.imread(large_path)
    small = tifffile.imread(small_path)
    
    if large is None or small is None:
        raise ValueError("One or both images could not be loaded.")
    
    # Determine dimensions (supporting grayscale and color images)
    if large.ndim == 2:
        H, W = large.shape
    else:
        H, W = large.shape[:2]
    
    if small.ndim == 2:
        h, w = small.shape
    else:
        h, w = small.shape[:2]
    
    # Approximate center of the large image
    center_x = W // 2
    center_y = H // 2

    best_score = float('inf')
    best_coord = (0, 0)
    best_diff = None

    # Search within ±search_radius pixels around the center
    for dx in range(-search_radius, search_radius + 1):
        for dy in range(-search_radius, search_radius + 1):
            top_left_x = center_x - w // 2 + dx
            top_left_y = center_y - h // 2 + dy
            
            # Check boundaries
            if top_left_x < 0 or top_left_y < 0 or top_left_x + w > W or top_left_y + h > H:
                continue
            
            # Extract the crop and compute the difference
            crop = large[top_left_y:top_left_y+h, top_left_x:top_left_x+w]
            # Convert to int32 to avoid underflow when subtracting, then take abs and convert back
            diff = np.abs(crop.astype(np.int32) - small.astype(np.int32)).astype(np.uint8)
            score = np.sum(diff)
            
            if score < best_score:
                best_score = score
                best_coord = (top_left_x, top_left_y)
                best_diff = diff

    # Ensure the output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Save the difference image with a filename that includes the crop coordinates and score
    diff_filename = f"diff_crop_x{best_coord[0]}_y{best_coord[1]}_score{best_score}.tiff"
    output_path = os.path.join(output_dir, diff_filename)
    tifffile.imwrite(output_path, best_diff)
    print("Saved difference image:", output_path)
    
    return best_coord, (w, h), best_score

# Example usage:
if __name__ == "__main__":
    large_img_path = r"C:\Users\Dolly\Downloads\temp\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag_0101_0.tif"  # Replace with your file path
    small_img_path = r"C:\Users\Dolly\Downloads\temp\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag_0101_1.tif"  # Replace with your file path
    output_dir = r"C:\Users\Dolly\Downloads\temp\output"  # Replace with your directory
    best_coord, crop_size, best_score = find_best_crop_save_diff(large_img_path, small_img_path, output_dir, search_radius=3)

    print("Best crop coordinate (top-left):", best_coord)
    print("Crop size (width, height):", crop_size)
    print("Difference score:", best_score)

Saved difference image: C:\Users\Dolly\Downloads\temp\output\diff_crop_x50_y50_score1216792903.tiff
Best crop coordinate (top-left): (50, 50)
Crop size (width, height): (3745, 3745)
Difference score: 1216792903


In [2]:
import os
import tifffile

def crop_tiff_directory(input_dir, output_dir, crop_coord, crop_size):
    """
    Crops all TIFF images in input_dir using the provided crop_coord (top-left) and crop_size (width, height)
    and saves them to output_dir.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for filename in os.listdir(input_dir):
        # Process only TIFF files
        if filename.lower().endswith(('.tif', '.tiff')):
            image_path = os.path.join(input_dir, filename)
            try:
                img = tifffile.imread(image_path)
            except Exception as e:
                print(f"Skipping {filename}: Unable to read file. Error: {e}")
                continue

            # Determine image dimensions
            if img.ndim == 2:  # Grayscale image
                H, W = img.shape
            elif img.ndim >= 3:  # Color or multi-dimensional image
                H, W = img.shape[:2]
            else:
                print(f"Skipping {filename}: Unsupported image dimensions.")
                continue

            top_left_x, top_left_y = crop_coord
            w, h = crop_size

            # Check if crop window is within image boundaries
            if top_left_x < 0 or top_left_y < 0 or top_left_x + w > W or top_left_y + h > H:
                print(f"Skipping {filename}: Crop dimensions out of bounds.")
                continue

            # Crop the image
            if img.ndim == 2:
                cropped = img[top_left_y:top_left_y + h, top_left_x:top_left_x + w]
            else:
                cropped = img[top_left_y:top_left_y + h, top_left_x:top_left_x + w, ...]

            # Save the cropped image as a TIFF
            output_path = os.path.join(output_dir, filename)
            try:
                tifffile.imwrite(output_path, cropped)
                print(f"Cropped and saved: {output_path}")
            except Exception as e:
                print(f"Error saving {filename}: {e}")

# Example usage:
if __name__ == "__main__":
    input_directory = r"C:\Users\Dolly\Documents\A.Study\data\kidney_left\2021_17\25.14um\recon\orig_recon_joseph\all\split_2\0\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag-0.01_0.92_"    # Replace with your TIFF images directory
    output_directory = r"C:\Users\Dolly\Documents\A.Study\data\kidney_left\2021_17\25.14um\recon\orig_recon_joseph\all\split_2\0\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag-0.01_0.92_crop_corrected"  # Replace with your output directory

    # Replace these with the crop coordinate and size obtained from Step 1.
    # For example, if your calibration yielded a top-left coordinate (50, 60)
    # and a crop size of (width=200, height=150):
    crop_coord = (50, 50)
    crop_size = (3745, 3745)

    crop_tiff_directory(input_directory, output_directory, crop_coord, crop_size)

Cropped and saved: C:\Users\Dolly\Documents\A.Study\data\kidney_left\2021_17\25.14um\recon\orig_recon_joseph\all\split_2\0\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag-0.01_0.92_crop_corrected\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag_0000.tif
Cropped and saved: C:\Users\Dolly\Documents\A.Study\data\kidney_left\2021_17\25.14um\recon\orig_recon_joseph\all\split_2\0\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag-0.01_0.92_crop_corrected\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag_0001.tif
Cropped and saved: C:\Users\Dolly\Documents\A.Study\data\kidney_left\2021_17\25.14um\recon\orig_recon_joseph\all\split_2\0\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag-0.01_0.92_crop_corrected\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag_0002.tif
Cropped and saved: C:\Users\Dolly\Documents\A.Study\data\kidney_left\2021_17\25.14um\recon\orig_recon_joseph\all\split_2\0\HA-900_25.14um_LADAF-2021-17_kidney_left_013__pag-0.01_0.92_crop_corrected\HA-900_25.14um_LADAF-2021-17_ki